# 03_concurrency_async — Concurrency & Async для ML Engineer

Этот ноутбук создан как **глубокий практический конспект для собеседований ML Engineer**.

Что внутри:
- подробная теория по конкурентности и асинхронности;
- практические Python-примеры;
- разбор производительности;
- вопросы в стиле интервью;
- практические задачи;
- типичные ошибки и анти-паттерны.

> Все объяснения — на русском, код — на Python.

## Как работать с ноутбуком

1. Читайте теорию и сразу запускайте код.
2. Смотрите на измерения времени (`time.perf_counter`).
3. Меняйте параметры (`N`, число потоков/процессов, задержки I/O).
4. Для интервью полезно уметь объяснить **почему** один подход быстрее другого в конкретной задаче.

In [ ]:
import asyncio
import concurrent.futures
import math
import os
import random
import threading
import multiprocessing as mp
import time
from dataclasses import dataclass
from pathlib import Path

## 1) GIL в деталях: внутреннее устройство и влияние на производительность

### 1. Теория

**GIL (Global Interpreter Lock)** в CPython — это глобальная блокировка, которая гарантирует, что в конкретный момент времени байткод Python исполняется только одним потоком.

Почему он появился исторически:
- упрощает управление памятью и reference counting;
- делает многие внутренние структуры интерпретатора проще и безопаснее;
- снижает сложность реализации расширений на C.

Как это проявляется:
- для **CPU-bound** Python-кода потоки обычно **не дают ускорения** на нескольких ядрах;
- для **I/O-bound** задач потоки эффективны, потому что во время I/O GIL часто освобождается.

Упрощённо про внутреннюю механику:
- интерпретатор периодически проверяет необходимость переключения потоков;
- поток, удерживающий GIL, исполняет байткод;
- другой поток может продолжить только после передачи GIL;
- C-расширения могут вручную освобождать GIL при долгих операциях (например, часть NumPy/библиотек ввода-вывода).

### 2. Код: CPU-bound в потоках vs одиночное исполнение

In [ ]:
def cpu_heavy(n: int) -> int:
    # Искусственная CPU-нагрузка на Python-циклах
    s = 0
    for i in range(n):
        s += (i * i) % 97
    return s


def run_single_thread(total_n: int):
    t0 = time.perf_counter()
    result = cpu_heavy(total_n)
    dt = time.perf_counter() - t0
    return result, dt


def run_with_threads(total_n: int, workers: int = 4):
    chunk = total_n // workers
    results = [0] * workers

    def worker(idx: int):
        results[idx] = cpu_heavy(chunk)

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(workers)]
    t0 = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    dt = time.perf_counter() - t0
    return sum(results), dt

N = 5_000_000
single_result, single_time = run_single_thread(N)
thread_result, thread_time = run_with_threads(N, workers=4)

print(f"Single: {single_time:.3f}s, result={single_result}")
print(f"Threads: {thread_time:.3f}s, result={thread_result}")

### 3. Подробное объяснение

- Даже при 4 потоках CPU-bound Python-код обычно не ускоряется пропорционально числу ядер.
- Причина: потоки конкурируют за GIL и фактически сериализуются на уровне интерпретатора.
- Иногда потоковая версия даже медленнее из-за overhead на создание/переключение потоков.

### 4. Производительность

- Для CPU-heavy pure Python используйте `multiprocessing`, C/C++ расширения, Numba, vectorized NumPy.
- Измеряйте время на вашей машине: эффект зависит от ОС, версии Python, характера кода.

### 5. Интервью-вопросы

1. Почему GIL мешает ускорять CPU-bound код потоками?
2. Почему при I/O задачах потоки всё же полезны?
3. Когда GIL «не проблема» в ML-пайплайне?

### 6. Практические задачи

- Модифицируйте пример: сравните `threading` и `multiprocessing` на одинаковой CPU-нагрузке.
- Увеличьте число потоков 1→2→4→8 и постройте таблицу времени.

### 7. Частые ошибки

- Уверенность, что «много потоков = всегда быстрее».
- Бенчмарк на слишком маленьких задачах (шум больше полезного сигнала).
- Игнорирование warm-up и кэш-эффектов.

## 2) Модуль `threading`

### 1. Теория

`threading` даёт параллелизм на уровне потоков внутри одного процесса.

Подходит для:
- сетевого I/O;
- ожидания файлов/сокетов;
- конкурентного orchestration вокруг внешних сервисов.

Не даёт true parallel исполнения Python-байткода для CPU-bound из-за GIL.

### 2. Код: I/O-bound имитация

In [ ]:
def io_task(task_id: int, delay: float = 0.5):
    time.sleep(delay)  # имитация I/O ожидания
    return f"task-{task_id} done"


def run_io_sequential(n: int = 6, delay: float = 0.5):
    t0 = time.perf_counter()
    results = [io_task(i, delay) for i in range(n)]
    return results, time.perf_counter() - t0


def run_io_threads(n: int = 6, delay: float = 0.5):
    results = [None] * n

    def worker(i):
        results[i] = io_task(i, delay)

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n)]
    t0 = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return results, time.perf_counter() - t0

seq_results, seq_time = run_io_sequential()
thr_results, thr_time = run_io_threads()

print(f"Sequential I/O time: {seq_time:.3f}s")
print(f"Threaded I/O time:   {thr_time:.3f}s")

### 3. Подробное объяснение

- В `sleep` поток не занят вычислениями, поэтому другие потоки могут работать.
- Поэтому I/O-задачи хорошо «прячут» время ожидания при помощи потоков.

### 4. Производительность

- Используйте `ThreadPoolExecutor`, если нужно удобное API пула потоков.
- Для тысяч коротких задач может быть критичен overhead планирования.

### 5. Интервью-вопросы

1. В чём отличие `threading.Thread` и `ThreadPoolExecutor`?
2. Как отлаживать deadlock в многопоточном коде?
3. Какие операции обычно освобождают GIL?

### 6. Практические задачи

- Перепишите пример на `concurrent.futures.ThreadPoolExecutor`.
- Добавьте таймауты и retry-политику для «сетевых» задач.

### 7. Частые ошибки

- Общие mutable-структуры без синхронизации.
- Отсутствие `join()` и потеря контроля завершения потоков.
- Слишком много потоков → деградация.

## 3) Модуль `multiprocessing`

### 1. Теория

`multiprocessing` запускает отдельные процессы, у каждого — свой интерпретатор и свой GIL.

Плюсы:
- настоящий параллелизм для CPU-bound;
- масштабирование по ядрам.

Минусы:
- IPC overhead (сериализация/передача данных);
- больше памяти;
- сложнее дебаг.

### 2. Код: CPU-bound через процессы

In [ ]:
def cpu_heavy_chunk(n: int) -> int:
    s = 0
    for i in range(n):
        s += (i * i) % 97
    return s


def run_with_processes(total_n: int, workers: int = 4):
    chunk = total_n // workers
    t0 = time.perf_counter()
    with mp.Pool(processes=workers) as pool:
        parts = pool.map(cpu_heavy_chunk, [chunk] * workers)
    dt = time.perf_counter() - t0
    return sum(parts), dt

if __name__ == "__main__":
    N = 5_000_000
    proc_result, proc_time = run_with_processes(N, workers=4)
    print(f"Multiprocessing: {proc_time:.3f}s, result={proc_result}")

### 3. Подробное объяснение

- Здесь задачи исполняются в разных процессах, поэтому ограничение GIL не мешает.
- Но есть цена: передача аргументов/результатов между процессами.

### 4. Производительность

- Эффективно для крупных CPU-задач.
- Для маленьких задач overhead IPC может «съесть» выигрыш.
- В ML: полезно для preprocessing, feature engineering, batch infer на CPU.

### 5. Интервью-вопросы

1. Почему `multiprocessing` часто быстрее `threading` на CPU-bound?
2. Что такое `spawn` vs `fork` и как это влияет на поведение?
3. Когда shared memory предпочтительнее pickle-передачи?

### 6. Практические задачи

- Добавьте сравнение времени для разных размеров `N`.
- Проверьте scaling по числу процессов до числа логических ядер.

### 7. Частые ошибки

- Передача огромных объектов в процессы без необходимости.
- Забывают guard `if __name__ == "__main__":`.
- Чрезмерный fan-out процессов.

## 4) CPU-bound vs I/O-bound задачи

### 1. Теория

- **CPU-bound**: упирается во вычисления CPU.
- **I/O-bound**: упирается в ожидание диска/сети/БД.

Правило выбора:
- CPU-bound → `multiprocessing` / векторизация / нативный код.
- I/O-bound → `threading` или `asyncio`.

### 2. Код: мини-классификатор задачи

In [ ]:
@dataclass
class BenchmarkResult:
    label: str
    elapsed: float


def benchmark(fn, label: str):
    t0 = time.perf_counter()
    fn()
    return BenchmarkResult(label=label, elapsed=time.perf_counter() - t0)

cpu_case = benchmark(lambda: cpu_heavy(3_000_000), "CPU-bound sample")
io_case = benchmark(lambda: time.sleep(0.8), "I/O-bound sample")

print(cpu_case)
print(io_case)

### 3. Подробное объяснение

Главный вопрос на интервью: **где bottleneck?**

Если bottleneck в CPU, нужно уменьшать число Python-операций или параллелить процессами.
Если bottleneck во внешних ожиданиях, нужно повышать конкурентность ожиданий.

### 4. Производительность

- Профилируйте (`cProfile`, sampling profilers, tracing).
- Не выбирайте технологию «на глаз».

### 5. Интервью-вопросы

1. Как определить, что задача I/O-bound?
2. Почему async особенно эффективен при высоком количестве сетевых запросов?
3. Что в ML чаще CPU-bound, а что I/O-bound?

### 6. Практические задачи

- Возьмите ваш preprocessing pipeline и выделите CPU/I/O этапы.
- Для каждого этапа предложите оптимальную модель конкурентности.

### 7. Частые ошибки

- Смешение CPU и I/O задач в одном и том же воркере без стратегии.
- Преждевременная оптимизация без профилирования.

## 5) Race conditions

### 1. Теория

Race condition возникает, когда итог зависит от недетерминированного порядка выполнения потоков/процессов.

Классический паттерн:
- read-modify-write общей переменной без синхронизации.

### 2. Код: гонка на инкременте

In [ ]:
counter = 0

def unsafe_increment(n_iters: int):
    global counter
    for _ in range(n_iters):
        counter += 1  # неатомарно на уровне логики

threads = [threading.Thread(target=unsafe_increment, args=(100_000,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Expected:", 400_000)
print("Actual:  ", counter)

### 3. Подробное объяснение

Даже если иногда значение «случайно» совпадает с ожидаемым, это не означает безопасность.
В реальных системах такие баги плавающие и трудно воспроизводимые.

### 4. Производительность

- Исправление гонок синхронизацией добавляет overhead, но это цена корректности.
- Для high-throughput систем применяют более тонкие структуры (очереди, lock-free на уровне нативного кода).

### 5. Интервью-вопросы

1. Почему race condition может не проявляться на каждом запуске?
2. Чем отличается thread-safe и deterministic?
3. Какие инструменты помогают ловить гонки?

### 6. Практические задачи

- Перепишите пример с `Lock` и сравните время.
- Сделайте потокобезопасный счётчик через очередь событий.

### 7. Частые ошибки

- Надежда на «везение планировщика».
- Частичная синхронизация (закрыли не все критические секции).

## 6) Lock, RLock, Semaphore

### 1. Теория

- `Lock`: базовая взаимоисключающая блокировка.
- `RLock`: reentrant lock, один и тот же поток может захватить несколько раз.
- `Semaphore`: ограничивает число одновременно работающих секций.

### 2. Код: примеры синхронизации

In [ ]:
lock = threading.Lock()
rlock = threading.RLock()
sema = threading.Semaphore(2)
safe_counter = 0


def safe_increment(n_iters: int):
    global safe_counter
    for _ in range(n_iters):
        with lock:
            safe_counter += 1


def nested_call(depth=2):
    with rlock:
        if depth > 0:
            nested_call(depth - 1)


def limited_worker(i):
    with sema:
        print(f"worker {i} entered")
        time.sleep(0.2)
        print(f"worker {i} left")

threads = [threading.Thread(target=safe_increment, args=(50_000,)) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("safe_counter:", safe_counter)
nested_call()

threads = [threading.Thread(target=limited_worker, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

### 3. Подробное объяснение

- `Lock` защищает критическую секцию.
- `RLock` нужен, когда функция с блокировкой рекурсивно вызывает себя или другой код, который использует тот же lock.
- `Semaphore(2)` допускает одновременно только двух workers.

### 4. Производительность

- Старайтесь делать критические секции минимальными.
- Избыточно грубый lock может убить параллелизм.

### 5. Интервью-вопросы

1. Когда нужен `RLock`, а не `Lock`?
2. Чем semaphore отличается от пула соединений концептуально?
3. Что такое contention и как его снижать?

### 6. Практические задачи

- Ограничьте число параллельных загрузок датасета до `k`.
- Добавьте измерение latency при разных значениях semaphore.

### 7. Частые ошибки

- Захват lock и долгий I/O внутри критической секции.
- Непоследовательный порядок захвата нескольких lock → deadlock.

## 7) Thread safety

### 1. Теория

Thread-safe код корректно работает при одновременном доступе из нескольких потоков.

Стратегии:
- immutable данные;
- message passing (очереди);
- минимизация shared state;
- явная синхронизация.

### 2. Код: producer/consumer через queue

In [ ]:
import queue

q = queue.Queue()
results = []
results_lock = threading.Lock()

def producer(n=10):
    for i in range(n):
        q.put(i)
    q.put(None)  # сигнал остановки


def consumer():
    while True:
        item = q.get()
        if item is None:
            break
        value = item * item
        with results_lock:
            results.append(value)

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)
p.start(); c.start()
p.join(); c.join()

print("results size:", len(results))
print("sample:", results[:5])

### 3. Подробное объяснение

`queue.Queue` потокобезопасна и снимает часть рисков ручной синхронизации.
Такой подход особенно полезен в ingestion пайплайнах ML.

### 4. Производительность

- Очереди удобны, но могут стать bottleneck при очень высокой нагрузке.
- Для heavy throughput стоит тестировать batch-передачу.

### 5. Интервью-вопросы

1. Какие структуры в стандартной библиотеке потокобезопасны?
2. Почему immutable-объекты упрощают thread safety?
3. Какой компромисс между безопасностью и latency?

### 6. Практические задачи

- Реализуйте multi-producer, multi-consumer схему.
- Добавьте graceful shutdown для нескольких consumers.

### 7. Частые ошибки

- Неполный протокол остановки воркеров.
- Доступ к общему списку без lock.

## 8) Основы `async/await`

### 1. Теория

`asyncio` реализует кооперативную конкурентность:
- `async def` создаёт coroutine;
- `await` добровольно отдаёт управление event loop;
- пока одна корутина ждёт I/O, исполняются другие.

Важно: async **не ускоряет CPU-bound Python-код сам по себе**.

### 2. Код: конкурентные корутины

In [ ]:
async def async_io_task(i: int, delay: float = 0.5):
    await asyncio.sleep(delay)
    return f"async-{i}-done"

async def run_async_tasks(n=6):
    t0 = time.perf_counter()
    tasks = [async_io_task(i, 0.5) for i in range(n)]
    results = await asyncio.gather(*tasks)
    dt = time.perf_counter() - t0
    return results, dt

results, dt = asyncio.run(run_async_tasks())
print("async results:", results[:3], "...")
print(f"async total time: {dt:.3f}s")

### 3. Подробное объяснение

- `await asyncio.sleep(...)` не блокирует поток целиком, а только текущую корутину.
- Поэтому много I/O задач можно обрабатывать в одном потоке эффективно.

### 4. Производительность

- Async эффективен при большом числе одновременно ожидающих операций.
- Следите за backpressure и ограничивайте конкурентность (`Semaphore`).

### 5. Интервью-вопросы

1. Чем coroutine отличается от thread?
2. Что происходит, если внутри coroutine выполнить блокирующий `time.sleep`?
3. Когда нужен `run_in_executor`?

### 6. Практические задачи

- Добавьте `asyncio.Semaphore` для ограничения числа параллельных запросов.
- Реализуйте retry с exponential backoff для асинхронной функции.

### 7. Частые ошибки

- Блокирующие вызовы в async-коде.
- «Забытый await» и невыполненные coroutines.

## 9) Event loop: концептуальные внутренности

### 1. Теория

Event loop — диспетчер задач, который:
1. хранит очередь готовых callback/coroutine continuation;
2. ожидает события I/O от ОС (селекторы/epoll/kqueue);
3. пробуждает соответствующие задачи;
4. выполняет их до следующего `await`.

Ключевая идея: **кооперативность**. Если задача не отдаёт управление (`await`), loop «зависает» для остальных.

### 2. Код: демонстрация блокировки loop

In [ ]:
async def bad_blocking_task():
    # ПЛОХО: блокирует event loop
    time.sleep(1.0)
    return "bad"

async def good_non_blocking_task():
    await asyncio.sleep(1.0)
    return "good"

async def compare_blocking_vs_non_blocking():
    t0 = time.perf_counter()
    await asyncio.gather(good_non_blocking_task(), good_non_blocking_task())
    non_block = time.perf_counter() - t0

    t1 = time.perf_counter()
    await asyncio.gather(bad_blocking_task(), bad_blocking_task())
    block = time.perf_counter() - t1

    return non_block, block

nb, b = asyncio.run(compare_blocking_vs_non_blocking())
print(f"non-blocking gather: {nb:.3f}s")
print(f"blocking gather:     {b:.3f}s")

### 3. Подробное объяснение

- В non-blocking случае две корутины делят время ожидания.
- В blocking случае `time.sleep` блокирует loop, задачи работают почти последовательно.

### 4. Производительность

- Любой блокирующий участок в event loop ухудшает latency всей системы.
- CPU-фрагменты выносите в process pool / thread pool (`run_in_executor`).

### 5. Интервью-вопросы

1. Почему один «плохой» coroutine может испортить throughput сервиса?
2. Чем event loop отличается от OS scheduler?
3. Что такое fairness в контексте loop?

### 6. Практические задачи

- Найдите и замените блокирующие вызовы в асинхронном коде.
- Добавьте таймауты на внешние I/O (`asyncio.wait_for`).

### 7. Частые ошибки

- Путаница между параллелизмом и конкурентностью.
- Непонимание, что async требует дисциплины `await`.

## 10) `asyncio.gather`

### 1. Теория

`asyncio.gather(*aws)` запускает awaitable-объекты конкурентно и возвращает результаты в исходном порядке.

Важные детали:
- если одна задача падает, по умолчанию исключение пробрасывается;
- `return_exceptions=True` позволяет собрать ошибки как результаты.

### 2. Код: обработка ошибок

In [ ]:
async def maybe_fail(i: int):
    await asyncio.sleep(0.1)
    if i % 3 == 0:
        raise ValueError(f"boom on {i}")
    return i * 10

async def gather_errors_demo():
    tasks = [maybe_fail(i) for i in range(1, 7)]

    try:
        await asyncio.gather(*tasks)
    except Exception as e:
        print("Default gather exception:", repr(e))

    tasks = [maybe_fail(i) for i in range(1, 7)]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results

print(asyncio.run(gather_errors_demo()))

### 3. Подробное объяснение

В production важно явно определить политику ошибок:
- fail-fast;
- частичный успех;
- retry для транзиентных ошибок.

### 4. Производительность

- `gather` удобен, но без лимита может породить слишком много одновременных задач.
- Для контроля используйте batching и semaphore.

### 5. Интервью-вопросы

1. Как меняется поведение `gather` с `return_exceptions=True`?
2. Когда лучше `as_completed`, чем `gather`?
3. Как ограничить конкурентность для 100k задач?

### 6. Практические задачи

- Реализуйте bounded-gather с semaphore.
- Добавьте retry только для сетевых ошибок.

### 7. Частые ошибки

- Безлимитный fan-out корутин.
- Потеря стека ошибок из-за неструктурированной обработки исключений.

## 11) `asyncio` vs `threading`

### 1. Теория

Сравнение:

- `threading`
  - проще интегрировать с legacy/blocking библиотеками;
  - дороже по памяти при очень большом числе задач;
  - для CPU-bound ограничен GIL.

- `asyncio`
  - эффективен для большого числа I/O операций в одном потоке;
  - требует async-экосистемы и дисциплины неблокирующего кода;
  - сложнее onboarding для команды без async-опыта.

### 2. Код: быстрый микро-бенч I/O

In [ ]:
def threaded_io_bench(n=100, delay=0.02):
    def task():
        time.sleep(delay)

    threads = [threading.Thread(target=task) for _ in range(n)]
    t0 = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return time.perf_counter() - t0


async def async_io_bench(n=100, delay=0.02):
    async def task():
        await asyncio.sleep(delay)

    t0 = time.perf_counter()
    await asyncio.gather(*[task() for _ in range(n)])
    return time.perf_counter() - t0

thread_time = threaded_io_bench()
async_time = asyncio.run(async_io_bench())

print(f"threading IO bench: {thread_time:.4f}s")
print(f"asyncio   IO bench: {async_time:.4f}s")

### 3. Подробное объяснение

На умеренных масштабах разница может быть небольшой.
На большом числе конкурентных I/O задач async часто выигрывает по управляемости и накладным расходам.

### 4. Производительность

- Выбор зависит от стека: если библиотека только blocking, async не даст автоматического чуда.
- В гибридных системах часто используют async + executor для blocking-фрагментов.

### 5. Интервью-вопросы

1. Когда бы вы выбрали threading вместо asyncio?
2. Как мигрировать legacy threading-сервис к async поэтапно?
3. Какие метрики сравнивать: throughput, p95 latency, memory?

### 6. Практические задачи

- Сделайте сравнительный benchmark для 100/1000/5000 задач.
- Добавьте ограничение конкурентности и сравните стабильность p95.

### 7. Частые ошибки

- «Религиозный» выбор технологии без учёта библиотек и команды.
- Сравнение без единых условий и корректной методики.

## 12) Когда async бесполезен

### 1. Теория

Async мало полезен или бесполезен, если:
- задача чисто CPU-bound в Python;
- вся критичная работа в блокирующей библиотеке без async API;
- concurrency очень низкая, и complexity async не окупается.

### 2. Код: async на CPU-bound

In [ ]:
async def fake_async_cpu():
    # Формально coroutine, но внутри CPU-работа без await
    s = 0
    for i in range(3_000_000):
        s += i % 7
    return s

async def run_fake_async_cpu():
    t0 = time.perf_counter()
    await asyncio.gather(fake_async_cpu(), fake_async_cpu())
    return time.perf_counter() - t0

print(f"fake async cpu time: {asyncio.run(run_fake_async_cpu()):.3f}s")

### 3. Подробное объяснение

Несмотря на `async`, тут нет точек кооперативного переключения, поэтому ускорения нет.

### 4. Производительность

- Для CPU: `multiprocessing`, vectorization, JIT, native extensions.
- Async оставьте orchestration-слою.

### 5. Интервью-вопросы

1. Почему `async def` сам по себе ничего не ускоряет?
2. Какие признаки того, что async в проекте «overengineering»?
3. Как аргументировать отказ от async на архитектурном ревью?

### 6. Практические задачи

- Возьмите CPU-этап из вашего проекта и реализуйте multiprocessing-версию.
- Сравните с «fake async» подходом по wall-clock.

### 7. Частые ошибки

- Переписывание CPU-кода в async без эффекта.
- Игнорирование стоимости поддержки сложной async-архитектуры.

## 13) Реальные ML-примеры

### 1. Теория

Типовые кейсы ML Engineer:

1. **Асинхронный сбор фичей** из нескольких сервисов (feature store, профили, ограничения).
2. **Параллельный preprocessing** CPU-тяжёлых трансформаций.
3. **Batch inference** с ограничением внешнего QPS.
4. **Гибрид**: async orchestration + process pool для CPU-этапов.

### 2. Код: async orchestration + CPU через executor

In [ ]:
def heavy_feature_transform(x: int) -> float:
    # CPU-этап
    acc = 0.0
    for i in range(200_000):
        acc += math.sqrt((x * i) % 97 + 1)
    return acc

async def fetch_feature_from_service(name: str, entity_id: int) -> float:
    # Имитируем сетевой вызов
    await asyncio.sleep(0.05 + random.random() * 0.05)
    return hash((name, entity_id)) % 100 / 10.0

async def build_features(entity_id: int):
    raw = await asyncio.gather(
        fetch_feature_from_service("profile", entity_id),
        fetch_feature_from_service("history", entity_id),
        fetch_feature_from_service("limits", entity_id),
    )
    return sum(raw)

async def score_entities(entity_ids):
    loop = asyncio.get_running_loop()
    scores = []

    for eid in entity_ids:
        base_feat = await build_features(eid)
        # CPU часть выносим в executor
        transformed = await loop.run_in_executor(None, heavy_feature_transform, int(base_feat * 10))
        scores.append((eid, transformed))

    return scores

entities = list(range(1, 6))
t0 = time.perf_counter()
sc = asyncio.run(score_entities(entities))
print(f"Scored {len(sc)} entities in {time.perf_counter()-t0:.3f}s")
print(sc[:2])

### 3. Подробное объяснение

- Сетевые вызовы к сервисам фичей — I/O, их удобно делать async.
- CPU-трансформация фичей вынесена через `run_in_executor`.
- В production можно использовать ProcessPool для тяжёлых CPU-функций.

### 4. Производительность

- Параметры, которые влияют сильнее всего:
  - размер батча;
  - лимиты конкурентности к внешним сервисам;
  - стоимость сериализации и формат данных.

### 5. Интервью-вопросы

1. Как построить online feature pipeline с ограничением QPS?
2. Где ставить кэш в async ML-сервисе?
3. Как изолировать медленные внешние зависимости?

### 6. Практические задачи

- Добавьте `asyncio.Semaphore` для лимита запросов к каждому сервису.
- Параллелизуйте scoring по батчам и снимите p95 latency.

### 7. Частые ошибки

- Отсутствие таймаутов и circuit breaker при внешних вызовах.
- Смешивание CPU и I/O без явного архитектурного разделения.

## 14) Чек-лист для собеседования ML Engineer

1. Умею объяснить GIL и его последствия.
2. Понимаю, когда threading, multiprocessing, asyncio.
3. Могу разобрать race condition и исправить через синхронизацию.
4. Понимаю ограничения event loop и опасность блокирующих вызовов.
5. Могу предложить архитектуру для реального ML pipeline с метриками производительности.

---

### Дополнительные упражнения (домашняя практика)

- Сделать мини-проект: async feature fetcher + process pool preprocess + batch inference.
- Добавить observability: время этапов, ошибки, p95/p99 latency.
- Написать короткий дизайн-док «почему выбран именно такой concurrency stack».